# Database Setup and Integration

Design the relational database schema and integrate
the processed recipe, ingredient and product datasets
into PostgreSQL.

## 1. Dataset Inspection

In [ ]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")

df_recipes = pd.read_parquet(PROCESSED_DIR / "recipes.parquet")
df_ingredients = pd.read_parquet(PROCESSED_DIR / "recipe_ingredients.parquet")
df_products = pd.read_parquet(PROCESSED_DIR / "products.parquet")

In [ ]:
for name, df in [
    ("Recipes", df_recipes),
    ("Ingredients", df_ingredients),
    ("Products", df_products)
]:
    print(f"\n{name}: {df.shape}")
    print(df.dtypes)

## 2. Database Schema Design

Define PostgreSQL tables, appropriate data types,
primary keys, foreign keys and integrity constraints.

In [ ]:
from uuid import UUID

def is_valid_uuid(value):
    try:
        UUID(str(value))
        return True
    except (ValueError, TypeError):
        return False

print("Valid recipe UUIDs:",
      df_recipes["recipe_id"].apply(is_valid_uuid).all())

print("Valid ingredient UUIDs:",
      df_ingredients["ingredient_id"].apply(is_valid_uuid).all())

print("\nCategories type:",
      type(df_recipes["categories"].iloc[0]))

print("Instructions type:",
      type(df_recipes["instructions"].iloc[0]))

print("\nExample categories:",
      df_recipes["categories"].iloc[0])

print("\nExample product IDs:")
display(df_products[["store", "id"]].head())

## 3. Python Database Connection

Establish and verify connectivity between the Python
application and the PostgreSQL database using Psycopg.

In [ ]:
import sys

sys.path.append("../src")

from eat_easy.db import get_connection

In [ ]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT current_database(), current_user;")
        result = cur.fetchone()

print("Database:", result[0])
print("User:", result[1])

## 4. Data Import

Populate PostgreSQL tables with the processed Parquet
datasets while preserving data types and relational integrity.

In [ ]:
df_recipes_import = df_recipes.copy()

for column in ["categories", "instructions"]:
    df_recipes_import[column] = (
        df_recipes_import[column].apply(list)
    )

recipe_rows = list(
    df_recipes_import.itertuples(index=False, name=None)
)

print("Prepared recipes:", len(recipe_rows))

In [ ]:
from psycopg import sql

columns = df_recipes_import.columns.tolist()

insert_query = sql.SQL("""
    INSERT INTO recipes ({})
    VALUES ({})
    ON CONFLICT (recipe_id) DO NOTHING
""").format(
    sql.SQL(", ").join(map(sql.Identifier, columns)),
    sql.SQL(", ").join(
        sql.Placeholder() for _ in columns
    )
)

In [ ]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.executemany(insert_query, recipe_rows)

        print("Inserted recipes:", cur.rowcount)

In [ ]:
with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM recipes;")
        print("Recipes in database:", cur.fetchone()[0])

In [ ]:
# Function for imports
def import_dataframe(df, table_name, conflict_columns):
    columns = df.columns.tolist()

    query = sql.SQL("""
        INSERT INTO {} ({})
        VALUES ({})
        ON CONFLICT ({}) DO NOTHING
    """).format(
        sql.Identifier(table_name),
        sql.SQL(", ").join(map(sql.Identifier, columns)),
        sql.SQL(", ").join(
            sql.Placeholder() for _ in columns
        ),
        sql.SQL(", ").join(
            map(sql.Identifier, conflict_columns)
        )
    )

    df_import = df.astype(object).where(pd.notna(df), None)
    rows = list(df_import.itertuples(index=False, name=None))

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.executemany(query, rows)
            inserted = cur.rowcount

    return inserted

In [ ]:
inserted = import_dataframe(
    df=df_ingredients,
    table_name="recipe_ingredients",
    conflict_columns=["recipe_id", "position"]
)

print("Inserted ingredients:", inserted)

In [ ]:
from decimal import Decimal

df_products_import = df_products.copy()

price_columns = [
    "price",
    "regular_price",
    "unit_price",
    "regular_unit_price"
]

for column in price_columns:
    df_products_import[column] = (
        df_products_import[column].apply(
            lambda x: Decimal(str(x)) if pd.notna(x) else None
        )
    )

In [ ]:
inserted = import_dataframe(
    df=df_products_import,
    table_name="products",
    conflict_columns=["store", "id"]
)

print("Inserted products:", inserted)

## 5. SQL Queries and Database Validation

Explore relational queries and validate database
functionality against Eat-easy's application requirements.

In [ ]:
def query_dataframe(query, params=None):
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(query, params)

            columns = [col.name for col in cur.description]
            rows = cur.fetchall()

    return pd.DataFrame(rows, columns=columns)

In [ ]:
query = """
    SELECT
        r.title,
        r.servings,
        i.name_lt,
        i.quantity,
        i.unit,
        i.total_grams
    FROM recipes AS r
    JOIN recipe_ingredients AS i
        ON r.recipe_id = i.recipe_id
    WHERE r.slug = %s
    ORDER BY i.position;
"""

query_dataframe(
    query,
    ("virtu-bulviu-cepelinai-su-mesa",)
)

In [ ]:
desired_servings = 2

query = """
    SELECT
        r.title,
        r.servings AS original_servings,
        i.name_lt,
        i.quantity AS original_quantity,
        i.unit,
        i.quantity * %s / r.servings AS scaled_quantity,
        i.total_grams * %s / r.servings AS scaled_grams
    FROM recipes AS r
    JOIN recipe_ingredients AS i
        ON r.recipe_id = i.recipe_id
    WHERE r.slug = %s
    ORDER BY i.position;
"""

df_scaled = query_dataframe(
    query,
    (
        desired_servings,
        desired_servings,
        "virtu-bulviu-cepelinai-su-mesa"
    )
)

display(df_scaled.round(2))

Mathematical portion scaling preserves ingredient ratios
but may produce impractical cooking quantities, particularly
for discrete ingredients such as eggs.

Practical quantity adjustments and purchasable package
rounding will be handled separately from mathematical scaling.

In [ ]:
selected_slugs = df_recipes["slug"].head(3).tolist()

display(
    df_recipes.loc[
        df_recipes["slug"].isin(selected_slugs),
        ["title", "servings"]
    ]
)

In [ ]:
query = """
    SELECT
        i.ingredient_id,
        i.name_lt,
        SUM(i.total_grams) AS required_grams,
        COUNT(DISTINCT r.recipe_id) AS recipe_count
    FROM recipes AS r
    JOIN recipe_ingredients AS i
        ON r.recipe_id = i.recipe_id
    WHERE r.slug = ANY(%s)
    GROUP BY i.ingredient_id, i.name_lt
    ORDER BY recipe_count DESC, required_grams DESC;
"""

df_shopping_list = query_dataframe(
    query,
    (selected_slugs,)
)

display(df_shopping_list.round(2))

In [ ]:
ingredient_name = "Sviestas"

query = """
    SELECT
        store,
        name,
        price,
        unit_price,
        unit,
        category
    FROM products
    WHERE name ILIKE %s
    ORDER BY unit_price ASC NULLS LAST
    LIMIT 20;
"""

df_matches = query_dataframe(
    query,
    (f"%{ingredient_name}%",)
)

display(df_matches)

In [ ]:
test_ingredients = [
    "Sviestas",
    "Sviestas (82% rieb.)",
    "Kiaušinis (virtas)",
    "Vištiena (krūtinėlė be odos)"
]

for ingredient in test_ingredients:
    matches = query_dataframe(
        query,
        (f"%{ingredient}%",)
    )

    print(f"\nIngredient: {ingredient}")
    print(f"Returned matches: {len(matches)}")

    display(matches[["store", "name"]].head(5))

## 6. Conclusions

PostgreSQL integration was completed successfully.

All three processed datasets were imported:
- 88 recipes
- 899 ingredient entries
- 11,086 grocery products

Relational queries, mathematical portion scaling and
ingredient aggregation were successfully tested.

Basic substring matching demonstrated limitations when
matching descriptive ingredient names to grocery products.

Further development will focus on semantic retrieval
using multilingual embeddings and systematic evaluation.